# ScopeOne Minimal Example

This notebook shows the minimum external workflow:

- import `ScopeOne`
- load a Micro-Manager config
- query available cameras
- query device properties
- start preview
- grab one frame into the notebook and display it inline
- stop preview and unload the config

Start `ScopeOne.exe` first. This notebook connects to the local ScopeOne API server.


In [ ]:
import sys
from pathlib import Path

import numpy as np
from IPython.display import display
from PIL import Image


def add_scopeone_python_path() -> tuple[Path, Path]:
    here = Path.cwd().resolve()
    candidates = [here, *here.parents]
    for base in candidates:
        if (base / "src" / "scopeone" / "__init__.py").exists() and (base / "pyproject.toml").exists():
            src_dir = base / "src"
            if str(src_dir) not in sys.path:
                sys.path.insert(0, str(src_dir))
            return base, base.parents[2]
    raise RuntimeError("Cannot find ScopeOne python project root")


def to_display_image(frame: np.ndarray) -> np.ndarray:
    frame = np.asarray(frame)
    if frame.dtype == np.uint8:
        return frame

    frame_min = int(frame.min())
    frame_max = int(frame.max())
    if frame_max <= frame_min:
        return np.zeros(frame.shape, dtype=np.uint8)

    scaled = (frame.astype(np.float32) - frame_min) / (frame_max - frame_min)
    return np.clip(scaled * 255.0, 0, 255).astype(np.uint8)


project_root, repo_root = add_scopeone_python_path()
config_path = repo_root / "config" / "MMConfig_demo.cfg"

from scopeone import ScopeOne

print("project_root:", project_root)
print("config_path:", config_path)
print("Make sure ScopeOne.exe is already running.")


In [ ]:
scopeone = ScopeOne()
scopeone.load_config(str(config_path))

camera_ids = scopeone.camera_ids()
print("cameras:", camera_ids)

if not camera_ids:
    raise RuntimeError("No cameras available after loading config")

camera_id = camera_ids[0]
print("using camera:", camera_id)


In [ ]:
property_names = scopeone.device_property_names(camera_id)
print("property count:", len(property_names))
print("first properties:", property_names[:10])

properties = scopeone.device_properties(camera_id)
for item in properties[:5]:
    print(item["name"], "=", item["value"], f"({item['type']})")

if "Exposure" in property_names:
    exposure = scopeone.get_property(camera_id, "Exposure", from_cache=False)
    print("Exposure:", exposure)

# To change a writable property, uncomment and choose a valid value:
# scopeone.set_property(camera_id, "Exposure", "10")


In [ ]:
scopeone.start_preview(camera_id)
print(f"Preview started for {camera_id}.")


In [ ]:
session = scopeone.record(frames=1, camera=camera_id)
frame = session.frame(camera_id, 0)

print("frame shape:", frame.shape)
print("frame dtype:", frame.dtype)

display_frame = to_display_image(frame)
display(Image.fromarray(display_frame))


## Stage Control

Read available stages, move Z by a small relative step, capture one frame, then restore the original Z position.


In [ ]:
print("XY stages:", scopeone.xy_stage_devices())
print("Z stages:", scopeone.z_stage_devices())

z_device = scopeone.current_focus_device()
z0 = scopeone.read_z_position(z_device)
print("focus device:", z_device)
print("z before:", z0)

try:
    scopeone.move_z_relative(-1.0, z_device)
    print("z after move:", scopeone.read_z_position(z_device))

    z_session = scopeone.record(frames=1, camera=camera_id)
    z_frame = z_session.frame(camera_id, 0)
    display(Image.fromarray(to_display_image(z_frame)))
finally:
    scopeone.move_z_to(z0, z_device)
    print("z restored:", scopeone.read_z_position(z_device))


## Minimal MDA

Capture one frame at each requested Z position. Use small demo offsets first.


In [ ]:
mda_session = scopeone.record(
    frames=8,
    camera=camera_id,
    z_positions=[z0, z0 + 1.0, z0],
    order=["time", "z"],
)
print("MDA cameras:", mda_session.camera_ids())
print("MDA frame count:", mda_session.frame_count(camera_id))
scopeone.move_z_to(z0, z_device)


In [ ]:
scopeone.stop_preview(camera_id)
scopeone.unload_config()
print(f"Preview stopped for {camera_id}.")
print("Configuration unloaded.")
